# Monte Carlo simulation in a scattering medium

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

import numpy as np
import matplotlib.pyplot as plt

from astropy import units
from numba import jit
from numpy import random

A rectangular box contains a medium with extinction $\alpha^a$ and $\alpha^s$ from absorption and scattering processes respectively. At the bottom of the box, photons are emitted vertically only from specific locations. The locations where photons are emitted are given by a mask saved in the file `RoCS_array.npy`, which contains an integer array. Photons should be emitted only from locations where this array is one. When plotting, be sure to use `origin='lower'` so that the orientation is correct:

In [ ]:
data = np.load("RoCS_array.npy")
fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(data, origin='lower', cmap=plt.get_cmap('gist_gray'));

You can assume the physical size of each pixel is 1 m x 1 m. 

By means of a Monte Carlo simulation, we can study how the scattering and absorption in the medium affect the intensity measured at the top of the box (after crossing the medium). We simulate photons being emitted in pixels given by the mask above, initially all in the vertical direction. After a photon travels an optical depth of $\tau_0$, it will interact with matter and suffer one of two outcomes: be absorbed (destroyed) or be scattered. Scattered photons should be given a random new direction of travel, and then travel a new optical depth of $\tau_1$ until they have another interaction with matter. This simulation of the photon trajectory should end when the photon is absorbed or it exits the box. Photons that exit through the sides or bottom are ignored, and photons that cross the top boundary are recorded. We use these to build an image of the intensity at the top of the box. 
    
<img style="float: right;" src="https://tiagopereira.space/ast4310/images/pattern.svg" width=300px>
    
The optical depth $\tau_i$ that a photon travels at each step $i$ should be probabilistically determined in the following way: $\tau_i = -\ln(\xi)$, where $\xi$ is a random number uniformly distributed in the interval (0, 1]. The type of interaction (absorption or scattering) will also be probabilistically determined, based on the photon destruction probability $\varepsilon$.

There are essentially three free parameters here: $\alpha^a$, $\alpha^s$, and the height of the box. Feel free to change them as you see fit. The main objective here is to study how the different parameters affect the intensity at the top of the box.

In [ ]:
"""
These functions are not completed. Fill in the missing code under TODO's
"""

# The jit macro compiles the code and speeds up the for-loops
@jit(nopython=True)
def monte_carlo(y, x, ny, nx, epsilon, alpha_tot, h, z=0):
    """
    Monte Carlo Simulation of photons going through a scattering medium. 
    Returns position of photon if it escapes.
    
    Parameters
    ----------
    x, y: float
        initital position of photon
    nx, ny: int
        pixel size of the box
    epsilon: float
        extinction coefficient
    alpha_tot: float
        total absorption
    h: float
        height of the box

    Returns:
        x, y: int
            pixel where photon escapes
        hit: bool/int
            0 if photon is destroyed, 1 if photon escapes
    """
    hit = 1
    phi = 0       # doesn't matter what phi is in the first iteration
    theta = 0     # photons emitted vertically
    
    while z < h:
        prob = random.uniform(0, 1)

        ### TODO
        # Find the length the photon will travel
        chi = ...
        tau = ...
        # l is the physical length the photon travels
        l = tau/alpha_tot

        # Calculate new coordinates
        x += l*np.sin(theta)*np.cos(phi)
        y += l*np.sin(theta)*np.sin(phi)
        z += l*np.cos(theta)

        # photon exits box
        if x < 0 or  y < 0 or z < 0 or y > ny or x > nx:
            hit = 0
            break

        # photon destruction
        elif epsilon > prob: 
            hit = 0
            break
        
        # photon scatters in random direction
        else:
            ### TODO
            # Assign a random direction to the photon
            phi = ...
            theta = ...
            continue
            
    return int(y), int(x), hit

@jit(nopython=True)
def calculate_scattering(image, alpha_a, alpha_s, h, N=50):
    """
    Calculates how the RoCS picture looks like through a medium

    Parameters
    ----------
    image: matrix
    alpha_a: float
        absorption coefficient
    alpha_s: float
        scattering coefficient
    h: float
        height of box
    N: int
        number of photons to simulate per pixel. Default is 50

    Returns
    -------
    diffuse: matrix
        diffused image
    """
    
    ny, nx = image.shape

    ### TODO
    # Calculate total extinction and destruction probability
    alpha_tot = ...
    epsilon = ...

    # output image
    diffuse_image = np.zeros(image.shape)

    # Loop over pixels that emit photons only
    non_zero_indices = np.argwhere(image != 0)
    for j, i in non_zero_indices:
        for k in range(N):
            y, x, hit = monte_carlo(j,i,ny,nx,epsilon,alpha_tot,h)
            diffuse_image[y, x] += hit
                    
    return diffuse_image

We plot for some different parameters:

In [ ]:
# Set random seed for reproducibility
random.seed(4310)

# Try with some different values for absorption coefficient while having fixed scattering
alpha_a = 1.0 # scattering coefficient
h = 20 # height of the box

rocs_pic1 = calculate_scattering(data,0.1,alpha_a,h)
random.seed(1)
rocs_pic2 = calculate_scattering(data,0.01,alpha_a,h)
random.seed(1)
rocs_pic3 = calculate_scattering(data,0.001,alpha_a,h)

# Plot the diffuse image
fig, ax = plt.subplots(2, 2, figsize=(11,6))
ax[0,0].set_title(r"Bottom of box")
ax[0,0].imshow(data/255,origin='lower',cmap=plt.get_cmap('gist_gray'))
ax[0,1].set_title(r"$\alpha^a = 0.1$")
ax[0,1].imshow(rocs_pic1,origin='lower',vmin=0,vmax=0.1,cmap=plt.get_cmap('binary_r'))
ax[1,0].set_title(r"$\alpha^a = 0.01$")
ax[1,0].imshow(rocs_pic2,origin='lower',vmin=0,vmax=0.1,cmap=plt.get_cmap('binary_r'))
ax[1,1].set_title(r"$\alpha^a = 0.001$")
ax[1,1].imshow(rocs_pic3,origin='lower',vmin=0,vmax=0.1,cmap=plt.get_cmap('binary_r'))
fig.tight_layout();